In [1]:
%pip install pandas numpy plotly ipywidgets nbformat


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import plotly.express as px
import numpy as np 
import unicodedata
from ipywidgets import interact, interactive, Dropdown, IntRangeSlider
from IPython.display import display, clear_output

#df = pd.read_csv('spotify-2023.csv', encoding='utf-8') #cannot read the whole file once bumping into errors
with open('spotify-2023.csv', mode='r', encoding='utf-8', errors='replace') as f:
    df = pd.read_csv(f)
df.head()

In [3]:
df = pd.read_csv('spotify-2023.csv', encoding='cp1252')
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


In [4]:
def fix_mojibake(text):
    if isinstance(text, str):
        try:
            # Re-encode back to bytes using latin1 and properly decode as utf-8
            return text.encode('latin-1').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            return text
    return text

df['artist(s)_name'] = df['artist(s)_name'].apply(fix_mojibake)
df['track_name'] = df['track_name'].apply(fix_mojibake)

In [5]:
def remove_accents(input_str):
    if not isinstance(input_str, str):
        return input_str
    # Decomposes accented characters into base letters + accent marks
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    # Filters out the accent marks
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

# Example: Converts "Bizarrap & Quevedo" special symbols cleanly
df['artist_clean'] = df['artist(s)_name'].apply(remove_accents)

In [6]:
df['artists_lst'] = df['artist(s)_name'].str.split(', ')
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%,artist_clean,artists_lst
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,Major,80,89,83,31,0,8,4,"Latto, Jung Kook","[Latto, Jung Kook]"
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,Major,71,61,74,7,0,10,4,Myke Towers,[Myke Towers]
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,Major,51,32,53,17,0,31,6,Olivia Rodrigo,[Olivia Rodrigo]
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,Major,55,58,72,11,0,11,15,Taylor Swift,[Taylor Swift]
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,Minor,65,23,80,14,63,11,6,Bad Bunny,[Bad Bunny]


In [7]:
df.describe()

,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,in_apple_playlists,in_apple_charts,in_deezer_charts,bpm,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
count,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.00000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000
mean,1.556139,2018.238195,6.033578,13.930745,5200.124869,12.009444,67.812172,51.908709,2.666317,122.540399,66.96957,51.431270,64.279119,27.057712,1.581322,18.213012,10.131165
std,0.893044,11.116218,3.566435,9.201949,7897.608990,19.575992,86.441493,50.630241,6.035599,28.057802,14.63061,23.480632,16.550526,25.996077,8.409800,13.711223,9.912888
min,1.000000,1930.000000,1.000000,1.000000,31.000000,0.000000,0.000000,0.000000,0.000000,65.000000,23.00000,4.000000,9.000000,0.000000,0.000000,3.000000,2.000000
25%,1.000000,2020.000000,3.000000,6.000000,875.000000,0.000000,13.000000,7.000000,0.000000,100.000000,57.00000,32.000000,53.000000,6.000000,0.000000,10.000000,4.000000
50%,1.000000,2022.000000,6.000000,13.000000,2224.000000,3.000000,34.000000,38.000000,0.000000,121.000000,69.00000,51.000000,66.000000,18.000000,0.000000,12.000000,6.000000
75%,2.000000,2022.000000,9.000000,22.000000,5542.000000,16.000000,88.000000,87.000000,2.000000,140.000000,78.00000,70.000000,77.000000,43.000000,0.000000,24.000000,11.000000
max,8.000000,2023.000000,12.000000,31.000000,52898.000000,147.000000,672.000000,275.000000,58.000000,206.000000,96.00000,97.000000,97.000000,97.000000,91.000000,97.000000,64.000000


In [8]:
df.iloc[0]['artists_lst']

['Latto', 'Jung Kook']

In [9]:
# check the list of artist names and their frequency in this top streamed dataset
artists = {}
for i in range(len(df)):
    names = df.iloc[i]['artists_lst']
    for name in names:
        if name in artists:
            artists[name] += 1
        else:
            artists[name] = 1    
print(artists)

{'Latto': 1, 'Jung Kook': 5, 'Myke Towers': 4, 'Olivia Rodrigo': 7, 'Taylor Swift': 38, 'Bad Bunny': 40, 'Dave': 2, 'Central Cee': 3, 'Eslabon Armado': 1, 'Peso Pluma': 16, 'Quevedo': 12, 'Gunna': 4, 'Yng Lvcas': 2, 'Grupo Frontera': 7, 'NewJeans': 6, 'Miley Cyrus': 2, 'David Kushner': 2, 'Harry Styles': 17, 'SZA': 23, 'Fifty Fifty': 2, 'Billie Eilish': 6, 'Feid': 21, 'Young Miko': 1, 'Jimin': 6, 'Gabito Ballesteros': 2, 'Junior H': 7, 'Arctic Monkeys': 4, 'Bizarrap': 10, 'The Weeknd': 37, 'Madonna': 1, 'Playboi Carti': 2, 'Fuerza Regida': 6, 'R��ma': 1, 'Selena G': 1, 'Tainy': 3, 'Morgan Wallen': 13, 'Dua Lipa': 9, 'Troye Sivan': 2, '21 Savage': 14, 'Metro Boomin': 14, 'Karol G': 10, 'Shakira': 6, 'Big One': 2, 'Duki': 5, 'Lit Killah': 2, 'Maria Becerra': 5, 'FMK': 2, 'Rusherking': 1, 'Emilia': 1, 'Tiago pzk': 5, 'Yahritza Y Su Esencia': 2, 'Post Malone': 7, 'Swae Lee': 3, 'Bebe Rexha': 1, 'David Guetta': 4, 'Tyler': 6, 'The Creator': 6, 'Kali Uchis': 3, 'Nicki Minaj': 6, 'Aqua': 1, '

In [10]:
num_artists = len(artists.keys())
num_artists

699

In [11]:
dict_to_list = list(artists.items())
dict_to_list

[('Latto', 1),
 ('Jung Kook', 5),
 ('Myke Towers', 4),
 ('Olivia Rodrigo', 7),
 ('Taylor Swift', 38),
 ('Bad Bunny', 40),
 ('Dave', 2),
 ('Central Cee', 3),
 ('Eslabon Armado', 1),
 ('Peso Pluma', 16),
 ('Quevedo', 12),
 ('Gunna', 4),
 ('Yng Lvcas', 2),
 ('Grupo Frontera', 7),
 ('NewJeans', 6),
 ('Miley Cyrus', 2),
 ('David Kushner', 2),
 ('Harry Styles', 17),
 ('SZA', 23),
 ('Fifty Fifty', 2),
 ('Billie Eilish', 6),
 ('Feid', 21),
 ('Young Miko', 1),
 ('Jimin', 6),
 ('Gabito Ballesteros', 2),
 ('Junior H', 7),
 ('Arctic Monkeys', 4),
 ('Bizarrap', 10),
 ('The Weeknd', 37),
 ('Madonna', 1),
 ('Playboi Carti', 2),
 ('Fuerza Regida', 6),
 ('R��ma', 1),
 ('Selena G', 1),
 ('Tainy', 3),
 ('Morgan Wallen', 13),
 ('Dua Lipa', 9),
 ('Troye Sivan', 2),
 ('21 Savage', 14),
 ('Metro Boomin', 14),
 ('Karol G', 10),
 ('Shakira', 6),
 ('Big One', 2),
 ('Duki', 5),
 ('Lit Killah', 2),
 ('Maria Becerra', 5),
 ('FMK', 2),
 ('Rusherking', 1),
 ('Emilia', 1),
 ('Tiago pzk', 5),
 ('Yahritza Y Su Esencia'

In [12]:
artists_df = pd.DataFrame(list(artists.items()), index=range(num_artists), columns=['artist', 'count'])
artists_df

,artist,count
0,Latto,1
1,Jung Kook,5
2,Myke Towers,4
3,Olivia Rodrigo,7
4,Taylor Swift,38
...,...,...
694,Mc Paiva ZS,1
695,Ludwig Goransson,1
696,Foudeqush,1
697,Jin,1


In [13]:
artists_df_sorted = artists_df.sort_values(by='count', ascending=False)
artists_df_sorted.head(10)

,artist,count
5,Bad Bunny,40
4,Taylor Swift,38
28,The Weeknd,37
116,Kendrick Lamar,23
18,SZA,23
21,Feid,21
165,Drake,19
17,Harry Styles,17
9,Peso Pluma,16
39,Metro Boomin,14


In [14]:
top_20_artists = artists_df_sorted.head(20)

In [15]:
fig = px.bar(
    top_20_artists, 
    x='count', 
    y='artist',
    orientation='h', 
    title='Top 20 Artists in Spotify 2023',
    text_auto=True
)

fig.show()

# check if there's any name typos
artists_df_sort_by_name = artists_df.sort_values(by='artist')
with pd.option_context('display.max_rows', None):
    display(artists_df_sort_by_name)

In [17]:
# check released_year unique values
print(df['released_year'].unique())

[2023 2019 2022 2013 2014 2018 2017 2020 2016 2012 1999 2008 1975 2021
 2015 2011 2004 1985 2007 2002 2010 1983 1992 1968 1984 2000 1997 1995
 2003 1973 1930 1994 1958 1957 1963 1959 1970 1971 1952 1946 1979 1950
 1942 1986 2005 1991 1996 1998 1982 1987]


In [18]:
len(df['released_year'].unique())

50

In [19]:
# check histogram of released_year
fig = px.histogram(
    df, 
    x='released_year',
    title='Frequency of released year of most streamed songs in 2023'
)

fig.show()

In [20]:
with pd.option_context('display.max_rows', None):
    display(df['released_year'].value_counts())

released_year
2022    402
2023    175
2021    119
2020     37
2019     36
2017     23
2016     18
2013     13
2014     13
2015     11
2018     10
2012     10
2011     10
2010      7
2002      6
1999      5
2004      4
1984      4
2000      4
1958      3
1963      3
2008      2
1975      2
1985      2
1995      2
2003      2
1957      2
1959      2
1970      2
1986      2
1991      2
1982      2
2007      1
1983      1
1992      1
1968      1
1997      1
1973      1
1930      1
1994      1
1971      1
1952      1
1946      1
1979      1
1950      1
1942      1
2005      1
1996      1
1998      1
1987      1
Name: count, dtype: int64

-> Released year between 2011 to 2022 yields 10 or more rows of data.

In [21]:
print(df.columns)

Index(['track_name', 'artist(s)_name', 'artist_count', 'released_year',
       'released_month', 'released_day', 'in_spotify_playlists',
       'in_spotify_charts', 'streams', 'in_apple_playlists', 'in_apple_charts',
       'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts', 'bpm',
       'key', 'mode', 'danceability_%', 'valence_%', 'energy_%',
       'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%',
       'artist_clean', 'artists_lst'],
      dtype='str')


In [22]:
df_2023 = df[df['released_year']==2023][['track_name', 'streams']]
df_2023.head(3)

,track_name,streams
0,Seven (feat. Latto) (Explicit Ver.),141381703
1,LALA,133716286
2,vampire,140003974


In [29]:
def top_stream_counts(df, year):
    this_year_df = df[df['released_year']==year][['track_name', 'artist(s)_name', 'streams']]
    this_year_sorted = this_year_df.sort_values(by='streams')
    top10 = this_year_sorted.head(10)
    fig = px.bar(
        top10,
        y = 'track_name',
        x = 'streams',
        hover_data='artist(s)_name',
        title = 'Top 10 most streamed songs released in ' + str(year),
        text_auto = True,
        orientation = 'h'
    )
    fig.show()

In [30]:
# let's test our bar chart
top_stream_counts(df, 2022)

In [31]:
def stream_vs_danceability(df, year):
    this_year_df = df[df['released_year']==year][['track_name', 'artist(s)_name', 'streams', 'released_month', 'danceability_%', 'energy_%']]

    fig = px.scatter(
        this_year_df, 
        x='danceability_%', 
        y='streams',
        color='released_month',                 
        size='energy_%',                       
        hover_name='track_name',          
        hover_data=['artist(s)_name'],     
        color_continuous_scale='Viridis',
        title='Streams vs. Danceability Percentage of most streamed songs released in ' + str(year)
    )

    fig.update_layout(template='plotly_white')
    fig.show()

In [32]:
#let's test scatterplot
stream_vs_danceability(df, 2023)

In [33]:
def released_month_hist(df, year):
    this_year_df = df[df['released_year']==year]['released_month']

    fig = px.histogram(
        this_year_df,
        x = 'released_month',
        nbins = 12,
        title = 'Number of most streamed songs released each month in ' + str(year)
    )
    fig.update_layout(
        xaxis_title='Released month in ' + str(year),
        yaxis_title='Number of most streamed songs in 2023',
        template='plotly_white'
    )
    fig.show()

In [34]:
released_month_hist(df, 2022)